# Feature Importance for ML Interviews

## Key Methods to Determine Feature Importance

### 1. **Tree-based Models (Built-in)**
- **Random Forest / XGBoost / LightGBM**: Use `feature_importances_` attribute
- Based on impurity reduction (Gini/entropy) or gain
- Pros: Fast, model-specific, handles non-linear relationships
- Cons: Can be biased toward high-cardinality features

### 2. **Permutation Importance**
- Shuffle feature values and measure performance drop
- Model-agnostic, works with any model
- More reliable than tree-based importance
- Implementation: `sklearn.inspection.permutation_importance`

### 3. **SHAP Values (SHapley Additive exPlanations)**
- Game theory approach: average marginal contribution
- Provides local (per-sample) and global importance
- Most interpretable, shows direction of impact
- Works with any model type

### 4. **Coefficient Magnitude (Linear Models)**
- For linear/logistic regression: absolute value of coefficients
- **Important**: Standardize features first for fair comparison
- Only works for linear relationships

### 5. **Mutual Information**
- Measures statistical dependence between features and target
- Model-agnostic, captures non-linear relationships
- Good for feature selection before modeling

### Interview Tips:
- **Always mention**: Permutation importance is more reliable than tree-based
- **For production**: Use SHAP for interpretability, permutation for validation
- **For feature selection**: Combine multiple methods for robustness
- **Scale matters**: For linear models, always standardize before comparing coefficients


In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import shap

# Example: Feature Importance Comparison
# Generate sample data
np.random.seed(42)
X = pd.DataFrame({
    'feature_1': np.random.randn(1000),
    'feature_2': np.random.randn(1000) * 2,  # Higher variance
    'feature_3': np.random.randn(1000) * 0.5,  # Lower variance
    'feature_4': np.random.randn(1000)
})
y = (X['feature_1'] + X['feature_2'] * 0.5 > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Method 1: Tree-based (Random Forest)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
tree_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Tree-based Importance:")
print(tree_importance)
print()

# Method 2: Permutation Importance (more reliable)
perm_importance = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)
print("Permutation Importance:")
print(perm_df)
print()

# Method 3: Linear Model Coefficients (must standardize!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
coef_importance = pd.Series(np.abs(lr.coef_[0]), index=X.columns).sort_values(ascending=False)
print("Linear Model Coefficients (absolute, standardized):")
print(coef_importance)
print()

# Method 4: SHAP Values (for tree models)
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test[:100])  # Sample for speed
# For binary classification, shap_values is a list [class_0_values, class_1_values]
# Use class 1 (positive class) for importance
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Use positive class for binary classification
# Compute mean absolute SHAP values across samples to get per-feature importance
# shap_values should be (n_samples, n_features), so mean(axis=0) gives (n_features,)
shap_importance_dict = {}
for i, col in enumerate(X.columns):
    shap_importance_dict[col] = np.abs(shap_values[:, i]).mean()
shap_importance = pd.Series(shap_importance_dict).sort_values(ascending=False)
print("SHAP Importance (mean absolute SHAP value):")
print(shap_importance)


Tree-based Importance:
feature_1    0.471174
feature_2    0.461748
feature_3    0.033608
feature_4    0.033470
dtype: float64

Permutation Importance:
     feature  importance_mean  importance_std
1  feature_2           0.3140        0.019723
0  feature_1           0.2925        0.029854
3  feature_4           0.0020        0.002449
2  feature_3           0.0005        0.001500

Linear Model Coefficients (absolute, standardized):
feature_2    5.896907
feature_1    5.845109
feature_4    0.075076
feature_3    0.025129
dtype: float64



/Users/junyishen/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/junyishen/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/junyishen/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


ValueError: Data must be 1-dimensional, got ndarray of shape (4, 2) instead